# 03 — Demand Forecasting

Runs the production `forecasting.train` on the cleaned daily
revenue series and visualises the actual vs forecast with the
confidence band. The 30-day holdout MAPE is the headline spec
metric for this model.

In [ ]:
import logging
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

from neuralretail.models.forecasting import train as fc_train, _mape, _prepare

logging.getLogger('neuralretail').setLevel(logging.WARNING)
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)

daily = pd.read_parquet('data/processed/daily_revenue.parquet').reset_index()
daily = daily.rename(columns={'InvoiceDate': 'ds', 'Revenue': 'y'})
print('daily shape:', daily.shape, '| range:', daily['ds'].min(), '→', daily['ds'].max())


## Fit Prophet with the same hyper-params as `make train`

Holdout = last 30 days; training = everything before.

In [ ]:
HORIZON = 30
train_df = daily.iloc[:-HORIZON].copy()
test_df = daily.iloc[-HORIZON:].copy()
print('train rows:', len(train_df), '| test rows:', len(test_df))

res = fc_train(train_df, horizon_days=0, run_name='nb_forecast')
future = res.model.make_future_dataframe(periods=HORIZON, freq='D')
fc = res.model.predict(future)
fc_test = fc.tail(HORIZON).set_index('ds')
mape = _mape(test_df.set_index('ds')['y'], fc_test['yhat'])
print(f'30-day holdout MAPE = {mape:.4f}')


## Actual vs forecast with 80% confidence interval

In [ ]:
ax = daily.set_index('ds')['y'].plot(label='actual', color='black', linewidth=1.0)
fc.set_index('ds')['yhat'].plot(ax=ax, label='forecast', color='coral')
ax.fill_between(
    fc['ds'], fc['yhat_lower'], fc['yhat_upper'],
    color='coral', alpha=0.2, label='80% CI',
)
ax.axvline(test_df['ds'].iloc[0], color='grey', linestyle='--', label='holdout start')
ax.set_title(f'Prophet forecast (30-day holdout MAPE = {mape:.4f})')
ax.set_ylabel('Revenue (GBP)')
ax.legend()
plt.tight_layout()
plt.show()
